<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/pod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title ⚙️ BƯỚC 1: CÀI ĐẶT THƯ VIỆN & TẢI MODEL AI
import os
from IPython.display import Audio, display, clear_output

print("⏳ Đang cài đặt thư viện (Chỉ mất 1-2 phút)...")
os.system('pip install -U qwen-tts huggingface_hub pydub')
os.system('apt-get install -y ffmpeg sox libsox-fmt-all')
clear_output()
print("✅ Cài đặt xong thư viện!")

from qwen_tts import Qwen3TTSModel
import torch
import soundfile as sf
import gc
import re
from pydub import AudioSegment

torch.backends.cudnn.benchmark = True
current_model = None
current_model_type = None

def load_model(task_type):
    global current_model, current_model_type
    model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign" if task_type == "DESIGN" else "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

    if current_model_type == task_type and current_model is not None:
        return current_model

    if current_model:
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"📥 Đang tải Model {task_type}... (Vui lòng đợi)")
    current_model = Qwen3TTSModel.from_pretrained(model_name, torch_dtype=torch.float16, device_map="cuda:0", attn_implementation="sdpa")
    current_model_type = task_type
    return current_model

print("✅ Hệ thống đã sẵn sàng!")

✅ Cài đặt xong thư viện!

********
********
 
✅ Hệ thống đã sẵn sàng!


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
# @title 🎙️ BƯỚC 2: TẠO GIỌNG MẪU NAM & NỮ

# --- BẠN CÓ THỂ SỬA TEXT Ở ĐÂY HOẶC GIỮ NGUYÊN ---
nam_prompt = "A professional male podcast host, deep and soothing voice. Laughing and energetic."
nam_text = "Haha! Hello everyone, welcome to the show."

nu_prompt = "A warm female voice, soft and velvety. American accent. Very happy."
nu_text = "Wow! I am so excited to be here."
# --------------------------------------------------

print("⏳ Đang tạo giọng Nam...")
model = load_model("DESIGN")
with torch.inference_mode():
    w_nam, sr_nam = model.generate_voice_design(text=nam_text, instruct=nam_prompt)
sf.write("nam_ref.wav", w_nam[0], sr_nam)

print("⏳ Đang tạo giọng Nữ...")
with torch.inference_mode():
    w_nu, sr_nu = model.generate_voice_design(text=nu_text, instruct=nu_prompt)
sf.write("nu_ref.wav", w_nu[0], sr_nu)

clear_output()
print("✅ Đã tạo xong 2 giọng mẫu!")
print("👨 Giọng Nam (nam_ref.wav):")
display(Audio("nam_ref.wav"))
print("👩 Giọng Nữ (nu_ref.wav):")
display(Audio("nu_ref.wav"))

In [ ]:
# @title 🚀 BƯỚC 3: RENDER PODCAST SIÊU TỐC

# 👇👇👇 DÁN KỊCH BẢN CỦA BẠN VÀO ĐÂY (Giữa 3 dấu ngoặc kép) 👇👇👇
kich_ban = """
Nam: Hello, everyone, and welcome back to the Just English Channel, the best place on the internet to practice your English listening skills. I am your host, David, and I am currently wearing the same sweatpants I have worn for three days.
Nu: And I am Emma. Welcome back, listeners. We are so happy to have you with us today. Please ignore David's terrible laundry habits. Today, we are going to talk about a very exciting and colorful topic. We are talking about shopping and fashion.
Nam: Fashion is a very strong word, Emma. I prefer to call it covering my body so I do not get arrested when I go outside to collect my pizza delivery.
Nu: That is exactly why we need to have this conversation, David. Fashion is not just about covering your body. It is a way to express your personality, boost your confidence, and show respect for the people around you.
Nam: I show respect by smelling like nice soap. I do not need a fancy suit to be a good person. Listeners, prepare yourselves for our first major debate of the day. Emma loves the shopping mall. I think the shopping mall is a terrifying nightmare of walking, bright lights, and spending money.
Nu: The shopping mall is a wonderful place! Going to the mall is a great physical activity. You walk around, you look at beautiful window displays, and you get out of the house. In English, we have a great phrase called window shopping.
Nam: Window shopping? Does that mean you go to a store just to buy glass windows for your house?
Nu: No, David. To window shop means to look at items in store windows without actually buying anything. It is a fun, relaxing way to spend a Saturday afternoon with your friends. You walk, you talk, you window shop, and then you drink some healthy green tea at the cafe.
Nam: That sounds like pure torture. Why would I walk for three kilometers just to look at things I am not going to buy? That is like looking at a picture of a delicious hamburger while eating a bowl of dry salad. It is just sad.
Nu: It is inspiring! You see the new trends, you feel the fabrics, and you try things on in the fitting room. You cannot do that from your sofa.
Nam: I absolutely can do that from my sofa. I am a master of online shopping. Online shopping is the greatest achievement of the modern world. I can buy a new winter jacket, a pair of sneakers, and a giant bag of potato chips at two o'clock in the morning while lying in my warm bed.
Nu: But when you shop online, you have no idea if the clothes will actually fit you! You just guess your size, click a button, and wait a week. Half the time, the clothes look completely different in real life.
Nam: That is why you buy three different sizes of the same shirt. Small, medium, and large. You try them all on in your bedroom, keep the one that fits, and send the other two back. It is a flawless system.
Nu: It is a terrible system for the environment because of all the shipping boxes! And be honest with me, David. Do you actually return the clothes that do not fit?
Nam: Well, returning things requires printing a label. And finding tape. And walking to the post office. So, no. I currently have a mountain of clothes in the corner of my bedroom that do not fit me. I call it the mountain of regret.
Nu: This is exactly why you need to go to a physical store! When you go to a shop, you take the clothes into the fitting room. You make sure they fit perfectly. We have a great idiom for this. When a piece of clothing fits you perfectly, you say it fits like a glove.
Nam: Fits like a glove. That makes sense, because a glove perfectly covers your hand. But I do not want my clothes to fit like a glove. I want my clothes to fit like a giant, comfortable blanket. Which brings me to our second heated debate, Emma. Let's talk about personal style.
Nu: Oh, I am ready for this. I believe in dressing for success.
Nam: And I believe in dressing for extreme comfort. Listeners, Emma is currently wearing a beautiful, perfectly ironed blazer, a nice blouse, and elegant shoes. We are recording a podcast. Nobody can even see us!
Nu: It does not matter if people can see me. Listeners, the phrasal verb to dress up means to wear nice, formal, or special clothes. I love to dress up. When I dress up, my brain knows it is time to work. It makes me feel professional, focused, and highly productive.
Nam: When I dress up, my brain thinks I am going to a wedding or a job interview, and I instantly feel stressed. My personal style is what I call the hoodie lifestyle.
Nu: The hoodie lifestyle? You mean wearing a baggy sweatshirt with a hood on it every single day?
Nam: Yes! A hoodie is a hug you can wear. It is soft, it is warm, and it has a big pocket in the front to hold my phone and my snacks. I pair my hoodie with loose sweatpants. It is the peak of human fashion evolution.
Nu: Sweatpants are for the gym, David. They are for exercising and sweating. They are not for walking around the city or going to work.
Nam: Excuse me, but sweatpants are for sitting on the sofa and playing video games. I never sweat in my sweatpants. That would ruin them. Plus, since the invention of video calls, I have the perfect strategy. Business on top, pajamas on the bottom.
Nu: That is so unprofessional! You wear a nice shirt for the camera, but you are secretly wearing pajama pants?
Nam: Absolutely. I call it the Zoom mullet. I look like a serious, professional businessman to my boss on the computer screen, but my legs are experiencing a relaxing tropical vacation in my fleece pajamas. It is the smartest fashion choice of the twenty-first century.
Nu: You are going to stand up during a meeting one day to get a glass of water, and your boss is going to see your cartoon pajama pants. It will be a recipe for disaster.
Nam: I will never stand up during a meeting. That requires physical effort. But let's talk about buying clothes. How much money do you spend on fashion? Because I refuse to spend a lot of money on fabric.
Nu: I actually do not buy a lot of clothes, but when I do, I buy high-quality pieces. I believe in having a capsule wardrobe.
Nam: A capsule? Like a spaceship? Are you wearing an astronaut suit?
Nu: No, David. For our B1 learners, a capsule wardrobe is a small collection of essential, high-quality clothes that never go out of style. You have a few good shirts, one perfect pair of jeans, a nice jacket, and they all match each other perfectly. You mix and match them.
Nam: That sounds incredibly boring. You wear the same five things every week?
Nu: It is not boring, it is elegant and sustainable! When you buy high-quality clothes, they last for years. You, on the other hand, buy fast fashion.
Nam: Fast fashion? Is that like fast food, but for clothes?
Nu: Exactly. Listeners, fast fashion refers to cheap, trendy clothing that is produced very quickly by mass-market retailers. They copy the latest trends, make them with cheap materials, and sell them for very low prices.
Nam: And what is wrong with that? Last week, I bought a neon green t-shirt online for exactly three dollars. Three dollars, Emma! I can buy a t-shirt for the price of a cup of coffee. I am a financial genius.
Nu: You are not a financial genius. Let's start our third debate. Fast fashion is terrible. What happened to that three-dollar neon green t-shirt after you washed it?
Nam: Well, I put it in the washing machine, and when I took it out, it was the size of a small cat. It shrank completely. Now I use it to clean my computer screen. But it only cost three dollars!
Nu: Exactly! You threw away three dollars. Fast fashion items fall apart after one wash. The buttons fall off, the zippers break, and the fabric shrinks. It is a huge waste of money, and it is devastating for the environment because millions of cheap shirts end up in the garbage every day.
Nam: But Emma, buying high-quality clothes like you do must cost a fortune. Listeners, to cost a fortune is a great idiom. It means something is extremely expensive. A nice designer jacket costs a fortune. I would rather buy ten cheap hoodies.
Nu: Quality does not always cost a fortune, David. You can wait for a sale, or you can go to a thrift store. A thrift store is a shop that sells second-hand or used clothes. You can find incredible, high-quality, vintage clothing there for a very low price.
Nam: A thrift store? You want me to wear someone else's old clothes? What if the previous owner was chased by a bear while wearing that jacket? It has bad luck attached to it.
Nu: You just wash it before you wear it! It is recycling. It is eco-friendly. You are just addicted to clicking the buy button on your phone. Are you a shopaholic, David?
Nam: A shopaholic?
Nu: Yes. A shopaholic is a person who loves shopping very much and cannot stop buying things. It is like an addiction.
Nu: You have a mountain of regret in your bedroom full of clothes you never returned, and you buy neon green shirts just because they are cheap. You might be a shopaholic.
Nam: I am absolutely not a shopaholic when it comes to real clothes. I hate buying real clothes. However, I might be a digital shopaholic.
Nu: What does that mean?
Nam: In my favorite video game, my digital character has a very extensive and expensive wardrobe. I spent twenty dollars last night buying a glowing blue armor suit for my digital wizard. He looks incredibly fashionable. He fits like a glove in that armor.
Nu: Let me get this straight. You refuse to spend more than three dollars on a t-shirt for your actual physical body, but you spend twenty dollars on digital clothes for a fake wizard?
Nam: My wizard is highly respected in the digital community, Emma. When he walks into a virtual village, people notice his style. When I walk into the kitchen to get pizza, my cat judges my sweatpants. It is about priorities.
Nu: You are completely hopeless. I am going to take you to a real clothing store next week and force you to try on a pair of actual jeans. Jeans with a zipper and a button. No elastic waistbands allowed.
Nam: That sounds like a physical prison for my legs. Please don't do that to me.
Nu: I am doing it for your own good. But before we go shopping, let's review the vocabulary and idioms we learned today so our listeners can practice.
Nam: Good idea. My brain is getting tired from all this talk about zippers and buttons. First, we learned the phrase window shopping. This is the terrible activity where you walk around a mall looking at things without buying anything.
Nu: It is a lovely activity, David. Then we learned the idiom fits like a glove. If a piece of clothing fits like a glove, it means it is exactly the right size and fits your body perfectly.
Nam: Next, we talked about the phrasal verb to dress up, which means to put on nice or formal clothes, usually to feel professional or go somewhere special. Emma loves to dress up. I prefer the hoodie lifestyle.
Nu: We also discussed the concept of fast fashion. This refers to cheap, poor-quality clothes that are made quickly to follow trends. They usually shrink or break very fast, just like David's neon green shirt.
Nam: Hey, that shirt is now a very high-quality cleaning cloth for my monitor. Then we learned the idiom to cost a fortune, which means to be very, very expensive.
Nu: And finally, we learned the word shopaholic, which describes someone who is addicted to shopping and buying things. Which perfectly describes David's video game habits.
Nam: My digital wizard is an icon of modern style! Anyway, it is time for our viewer challenge. Listeners, we want to hear from you in the comments section below.
Nu: Yes! Tell us about your personal shopping and fashion style. Are you firmly Team Emma? Do you love going to physical stores, building a high-quality capsule wardrobe, and dressing up to feel productive?
Nam: Or are you proudly Team David? Do you shop online at two in the morning, embrace the extreme comfort of sweatpants and hoodies, and rock the business-on-top, pajamas-on-the-bottom look for video calls? Let us know in the comments. We all know my way is much more comfortable.
Nu: Please ignore his fashion advice. We absolutely love reading your comments and practicing English with you all. If you enjoyed our funny debates today, please hit the like button, subscribe to the Just English Channel, and turn on the notification bell so you never miss an episode.
Nam: And remember, you can hit that subscribe button while wearing your pajamas. We cannot see you, and we will not judge you.
Nu: Thank you so much for tuning in and listening, everyone. Keep practicing your English daily, maybe clean out your closet this weekend, and we will see you in the next exciting episode!
Nam: Goodbye everyone! I am going to go buy a digital hat for my video game character now. He needs an accessory for his glowing blue armor.
Nu: You need a real pair of pants, David. Have a wonderful week, listeners!
"""
# 👆👆👆 ========================================================== 👆👆👆

file_nam = "nam_ref.wav"
file_nu = "nu_ref.wav"

if not os.path.exists(file_nam) or not os.path.exists(file_nu):
    print("⚠️ LỖI: Không tìm thấy file giọng mẫu. Hãy chạy Bước 2 trước!")
else:
    model = load_model("CLONE")
    final_audio = AudioSegment.silent(duration=500)
    gap = AudioSegment.silent(duration=400)

    lines = [l for l in kich_ban.strip().split('\n') if ":" in l]
    total_lines = len(lines)

    print("⚡️ Đang học giọng Nam & Nữ (Chỉ làm 1 lần)...")
    nam_prompt_feature = model.create_voice_clone_prompt(ref_audio=file_nam, ref_text=None, x_vector_only_mode=True)
    nu_prompt_feature = model.create_voice_clone_prompt(ref_audio=file_nu, ref_text=None, x_vector_only_mode=True)

    print("🚀 BẮT ĐẦU THU ÂM PODCAST...")
    for index, line in enumerate(lines):
        name_part, text = line.split(":", 1)
        clean_name = re.sub(r'\(.*?\)', '', name_part).strip().lower()
        text = text.strip()

        # Nhận diện Nam/Nữ (Đã cập nhật đủ chữ david, emma)
        is_nam = clean_name in ["nam", "man", "male", "host", "teacher", "eric", "ryan", "mr", "david"]
        current_prompt = nam_prompt_feature if is_nam else nu_prompt_feature

        if text:
            print(f"🎙️ [{index+1}/{total_lines}] {clean_name.upper()}: Đang thu âm...")
            with torch.inference_mode():
                w, sr = model.generate_voice_clone(text=text, voice_clone_prompt=current_prompt)

            sf.write("temp.wav", w[0], sr)
            final_audio += AudioSegment.from_wav("temp.wav") + gap
            if os.path.exists("temp.wav"): os.remove("temp.wav")

    # Xuất file
    final_audio.export("Podcast_ThanhPham.mp3", format="mp3")
    clear_output()
    print("🎉 HOÀN TẤT! Đã ghép nối xong toàn bộ kịch bản.")
    print("👇 Bấm Play để nghe hoặc bấm dấu 3 chấm (⋮) để Tải xuống MP3 👇")
    display(Audio("Podcast_ThanhPham.mp3"))

🎉 HOÀN TẤT! Đã ghép nối xong toàn bộ kịch bản.
👇 Bấm Play để nghe hoặc bấm dấu 3 chấm (⋮) để Tải xuống MP3 👇
